# Assignment 2: CNN Representations and Color Disk Prediction

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rrfhwn/neural-architectures-and-representation-learning-course/blob/main/weeks/07/Assignment_02_CNN_Representations.ipynb)

**Course:** Neural Architectures and Representation Learning  
**Related notebook:** `Week_07_CNNs_Visual_Representations.ipynb`


## Task

You will use a CNN representation to predict a simple visual property of color images.

The assignment uses CIFAR-10 images. For each image, we compute the average RGB color, convert it to hue and saturation, and place it on a disk:

```text
x = saturation * cos(hue)
y = saturation * sin(hue)
```

Your job is to train and compare small heads that predict this 2D disk coordinate from a frozen CNN representation.

The goal is not to build the best color predictor. The goal is to explain what a pretrained representation makes easy or difficult.


## Submission requirements

Submit a completed copy of this notebook.

Your submission must include:

1. The provided baseline run.
2. At least **three representation-head experiments**.
3. A final chosen head.
4. Train/validation MSE plots.
5. True vs predicted color-disk plots.
6. Example images with low and high prediction error.
7. Short written answers explaining your diagnosis and choices.

Do not change the fixed dataset split cell. This keeps submissions comparable.


## Grading rubric

| Criterion | Points |
|----------|--------|
| Correctly runs the fixed dataset and baseline | 15 |
| Performs controlled representation-head experiments | 25 |
| Uses clear plots and error analysis | 25 |
| Chooses and justifies a final model | 20 |
| Explains what the representation captures and misses | 15 |

Total: 100 points.


---

## Environment

This notebook uses `torch`, `torchvision`, `numpy`, and `matplotlib`. CPU is enough, but the pretrained-weight download may take a little time.


In [ ]:
import math
import random
import colorsys
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except Exception:
    plt.rc("axes", grid=True)

%matplotlib inline

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
print("torch:", torch.__version__)

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(72)


---

## 1. Fixed setup

Do not modify this section. Everyone uses the same image subset, feature extractor setup, baseline metric, and target definition.


In [ ]:
def rgb_to_disk_coordinates(rgb_array):
    coords = []
    hsv_values = []
    for r, g, b in rgb_array:
        h, s, v = colorsys.rgb_to_hsv(float(r), float(g), float(b))
        angle = 2 * math.pi * h
        coords.append([s * math.cos(angle), s * math.sin(angle)])
        hsv_values.append([h, s, v])
    return np.array(coords, dtype=np.float32), np.array(hsv_values, dtype=np.float32)


def disk_to_rgb(xy):
    colors = []
    for x, y in xy:
        s = min(1.0, math.sqrt(float(x * x + y * y)))
        h = (math.atan2(float(y), float(x)) / (2 * math.pi)) % 1.0
        colors.append(colorsys.hsv_to_rgb(h, s, 0.95))
    return np.array(colors)


def plot_disk(coords, title, ax=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(5, 5))
    ax.add_patch(plt.Circle((0, 0), 1.0, color="black", fill=False, linewidth=1.0, alpha=0.4))
    ax.scatter(coords[:, 0], coords[:, 1], c=disk_to_rgb(coords), s=24, alpha=0.8, edgecolor="black", linewidth=0.2)
    ax.axhline(0, color="black", linewidth=0.5, alpha=0.4)
    ax.axvline(0, color="black", linewidth=0.5, alpha=0.4)
    ax.set_xlim(-1.05, 1.05)
    ax.set_ylim(-1.05, 1.05)
    ax.set_aspect("equal")
    ax.set_xlabel("saturation * cos(hue)")
    ax.set_ylabel("saturation * sin(hue)")
    ax.set_title(title)
    return ax


In [ ]:
cifar_transform = transforms.Compose([transforms.ToTensor()])

try:
    cifar_train_full = datasets.CIFAR10(DATA_DIR, train=True, download=True, transform=cifar_transform)
    cifar_val_full = datasets.CIFAR10(DATA_DIR, train=False, download=True, transform=cifar_transform)
    class_names = cifar_train_full.classes
    print("Loaded CIFAR-10.")
except Exception as exc:
    print("CIFAR-10 download/load failed; using torchvision FakeData fallback.")
    print("Reason:", repr(exc))
    cifar_train_full = datasets.FakeData(size=1200, image_size=(3, 32, 32), num_classes=10, transform=cifar_transform)
    cifar_val_full = datasets.FakeData(size=600, image_size=(3, 32, 32), num_classes=10, transform=cifar_transform)
    class_names = [f"class_{i}" for i in range(10)]

# Fixed subset sizes for comparable submissions.
TRAIN_N = 600
VAL_N = 300
cifar_train = Subset(cifar_train_full, list(range(min(TRAIN_N, len(cifar_train_full)))))
cifar_val = Subset(cifar_val_full, list(range(min(VAL_N, len(cifar_val_full)))))

train_loader = DataLoader(cifar_train, batch_size=64, shuffle=False)
val_loader = DataLoader(cifar_val, batch_size=64, shuffle=False)
print("train images:", len(cifar_train))
print("validation images:", len(cifar_val))


In [ ]:
def show_samples(dataset, n=12):
    plt.figure(figsize=(10, 3))
    for i in range(n):
        img, label = dataset[i]
        plt.subplot(2, n // 2, i + 1)
        plt.imshow(img.permute(1, 2, 0).numpy())
        plt.title(class_names[label], fontsize=9)
        plt.axis("off")
    plt.suptitle("Fixed CIFAR-10 subset samples")
    plt.tight_layout()
    plt.show()

show_samples(cifar_train)


In [ ]:
@torch.no_grad()
def compute_targets(loader):
    coords_all = []
    images_all = []
    labels_all = []
    for X, y in loader:
        avg_rgb = X.mean(dim=(2, 3)).numpy()
        coords, _ = rgb_to_disk_coordinates(avg_rgb)
        coords_all.append(torch.tensor(coords, dtype=torch.float32))
        images_all.append(X)
        labels_all.append(y)
    return torch.cat(coords_all), torch.cat(images_all), torch.cat(labels_all)

train_targets, train_images, train_labels = compute_targets(train_loader)
val_targets, val_images, val_labels = compute_targets(val_loader)

plot_disk(train_targets.numpy(), "Training targets: average hue/saturation disk")
plt.tight_layout()
plt.show()


---

## 2. Frozen CNN representation

This section builds a frozen ResNet18 feature extractor. The assignment uses the vector before the final classifier as the representation.

If pretrained weights cannot be downloaded, the code falls back to random weights. Note this in your reflection if it happens, because an untrained representation is not equivalent to a pretrained representation.


In [ ]:
def build_feature_extractor(pretrained=True):
    try:
        weights = models.ResNet18_Weights.DEFAULT if pretrained else None
        model = models.resnet18(weights=weights)
        transform = weights.transforms() if weights is not None else transforms.Compose([
            transforms.Resize((96, 96)),
            transforms.Normalize(mean=(0.4914, 0.4822, 0.4465), std=(0.2470, 0.2435, 0.2616)),
        ])
        if weights is not None:
            print("Loaded ImageNet-pretrained ResNet18.")
    except Exception as exc:
        print("Could not load pretrained weights; using random ResNet18.")
        print("Reason:", repr(exc))
        model = models.resnet18(weights=None)
        transform = transforms.Compose([
            transforms.Resize((96, 96)),
            transforms.Normalize(mean=(0.4914, 0.4822, 0.4465), std=(0.2470, 0.2435, 0.2616)),
        ])
    feature_dim = model.fc.in_features
    model.fc = nn.Identity()
    model.eval().to(device)
    for p in model.parameters():
        p.requires_grad = False
    return model, transform, feature_dim

feature_extractor, feature_transform, feature_dim = build_feature_extractor(pretrained=True)
print("feature_dim:", feature_dim)


In [ ]:
@torch.no_grad()
def extract_features(loader, model, transform):
    features = []
    for X, _ in loader:
        X = transform(X).to(device)
        features.append(model(X).cpu())
    return torch.cat(features)

train_features = extract_features(train_loader, feature_extractor, feature_transform)
val_features = extract_features(val_loader, feature_extractor, feature_transform)
print("train features:", tuple(train_features.shape))
print("validation features:", tuple(val_features.shape))


---

## 3. Baseline: predict the mean disk coordinate

A useful model should beat this baseline. The baseline ignores the image and predicts the same average color point for every validation image.


In [ ]:
mean_prediction = train_targets.mean(dim=0, keepdim=True)
baseline_val_pred = mean_prediction.repeat(len(val_targets), 1)
baseline_mse = torch.mean((baseline_val_pred - val_targets) ** 2).item()
print("baseline validation MSE:", baseline_mse)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
plot_disk(val_targets.numpy(), "True validation disk", axes[0])
plot_disk(baseline_val_pred.numpy(), "Baseline predictions", axes[1])
plt.tight_layout()
plt.show()


### Baseline diagnosis

Write 2-3 sentences:

- Why is this a weak baseline?
- What would a representation-based model need to learn to beat it?


---

## 4. Representation-head utilities

You may change head settings in later experiment cells. Do not change the utility functions unless you clearly explain why.


In [ ]:
class ColorDiskHead(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, depth=1, dropout=0.0):
        super().__init__()
        layers = []
        current = input_dim
        for _ in range(depth):
            layers.append(nn.Linear(current, hidden_dim))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            current = hidden_dim
        layers.append(nn.Linear(current, 2))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


def train_head(config, seed=72):
    set_seed(seed)
    head = ColorDiskHead(
        train_features.shape[1],
        hidden_dim=config.get("hidden_dim", 64),
        depth=config.get("depth", 1),
        dropout=config.get("dropout", 0.0),
    ).to(device)
    optimizer = torch.optim.Adam(
        head.parameters(),
        lr=config.get("lr", 1e-3),
        weight_decay=config.get("weight_decay", 0.0),
    )
    loss_fn = nn.MSELoss()
    X_train = train_features.to(device)
    y_train = train_targets.to(device)
    X_val = val_features.to(device)
    y_val = val_targets.to(device)
    history = {"train_mse": [], "val_mse": []}
    for epoch in range(config.get("epochs", 80)):
        head.train()
        optimizer.zero_grad()
        loss = loss_fn(head(X_train), y_train)
        loss.backward()
        optimizer.step()

        head.eval()
        with torch.no_grad():
            val_loss = loss_fn(head(X_val), y_val)
        history["train_mse"].append(float(loss.item()))
        history["val_mse"].append(float(val_loss.item()))
    return head, history


@torch.no_grad()
def predict(head, features):
    head.eval()
    return head(features.to(device)).cpu()


def plot_history(history, title):
    plt.figure(figsize=(6, 4))
    plt.plot(history["train_mse"], label="train")
    plt.plot(history["val_mse"], label="validation")
    plt.axhline(baseline_mse, color="black", linestyle="--", label="mean baseline")
    plt.yscale("log")
    plt.xlabel("epoch")
    plt.ylabel("MSE")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()


def plot_prediction_disks(pred, title):
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    plot_disk(val_targets.numpy(), "True validation disk", axes[0])
    plot_disk(pred.numpy(), title, axes[1])
    plt.tight_layout()
    plt.show()


def show_error_examples(pred, n=12):
    pred_np = pred.numpy()
    true_np = val_targets.numpy()
    errors = np.linalg.norm(pred_np - true_np, axis=1)
    order = np.argsort(errors)
    picks = np.concatenate([order[: n // 2], order[-(n // 2):]])
    plt.figure(figsize=(12, 4))
    for j, idx in enumerate(picks):
        img = val_images[idx].permute(1, 2, 0).numpy()
        plt.subplot(2, n // 2, j + 1)
        plt.imshow(img)
        plt.title(f"{class_names[int(val_labels[idx])]}\nerr={errors[idx]:.2f}", fontsize=9)
        plt.axis("off")
    plt.suptitle("Low-error examples first, high-error examples second")
    plt.tight_layout()
    plt.show()


---

## 5. Experiment 1

Start with a simple one-hidden-layer head.


In [ ]:
# TODO: you may edit this config, but keep a note of what changed.
experiment_1 = {
    "hidden_dim": 64,
    "depth": 1,
    "dropout": 0.0,
    "lr": 1e-3,
    "weight_decay": 0.0,
    "epochs": 80,
}

head_1, history_1 = train_head(experiment_1)
pred_1 = predict(head_1, val_features)
print("Experiment 1 final val MSE:", history_1["val_mse"][-1])
plot_history(history_1, "Experiment 1")
plot_prediction_disks(pred_1, "Experiment 1 predictions")
show_error_examples(pred_1)


### Experiment 1 notes

Write 2-4 sentences:

- What did you configure?
- Did it beat the mean baseline?
- What does the disk plot show?


---

## 6. Experiment 2

Change one thing. Good options: hidden dimension, learning rate, weight decay, depth, or dropout.


In [ ]:
# TODO: change exactly one or two settings compared with Experiment 1.
experiment_2 = {
    "hidden_dim": 128,
    "depth": 1,
    "dropout": 0.0,
    "lr": 1e-3,
    "weight_decay": 0.0,
    "epochs": 80,
}

head_2, history_2 = train_head(experiment_2)
pred_2 = predict(head_2, val_features)
print("Experiment 2 final val MSE:", history_2["val_mse"][-1])
plot_history(history_2, "Experiment 2")
plot_prediction_disks(pred_2, "Experiment 2 predictions")
show_error_examples(pred_2)


### Experiment 2 notes

Write 2-4 sentences:

- What changed from Experiment 1?
- Did validation improve?
- Did training and validation move together or split apart?


---

## 7. Experiment 3

Try a regularized or deeper head.


In [ ]:
# TODO: try a controlled third experiment.
experiment_3 = {
    "hidden_dim": 64,
    "depth": 2,
    "dropout": 0.10,
    "lr": 1e-3,
    "weight_decay": 1e-4,
    "epochs": 80,
}

head_3, history_3 = train_head(experiment_3)
pred_3 = predict(head_3, val_features)
print("Experiment 3 final val MSE:", history_3["val_mse"][-1])
plot_history(history_3, "Experiment 3")
plot_prediction_disks(pred_3, "Experiment 3 predictions")
show_error_examples(pred_3)


### Experiment 3 notes

Write 2-4 sentences:

- What changed?
- Was the change helpful?
- What tradeoff do you notice?


---

## 8. Compare experiments

Use the table and plots to choose a final model.


In [ ]:
results = {
    "baseline_mean": baseline_mse,
    "experiment_1": history_1["val_mse"][-1],
    "experiment_2": history_2["val_mse"][-1],
    "experiment_3": history_3["val_mse"][-1],
}

for name, mse in sorted(results.items(), key=lambda item: item[1]):
    print(f"{name:>16}: {mse:.5f}")

plt.figure(figsize=(7, 4))
plt.bar(results.keys(), results.values())
plt.ylabel("validation MSE")
plt.title("Experiment comparison")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()


### Comparison notes

Write 3-5 sentences:

- Which experiment performed best?
- Which experiment is easiest to justify?
- Is the best validation MSE also the most convincing model?


---

## 9. Final chosen head

Set `final_head` and `final_pred` to the model you choose. You may reuse one experiment or train one final controlled variant.


In [ ]:
# TODO: choose your final model.
# Example: reuse Experiment 2.
final_name = "experiment_2"
final_head = head_2
final_history = history_2
final_pred = pred_2

print("Final model:", final_name)
plot_history(final_history, f"Final model: {final_name}")
plot_prediction_disks(final_pred, f"Final predictions: {final_name}")
show_error_examples(final_pred)


## Final reflection

Answer in 6-10 sentences:

1. What representation did you use?
2. What did the small head learn?
3. Which configuration worked best and why?
4. Which images were hard, and why?
5. What does this tell you about CNN representations?
6. What would you try if you were allowed to fine-tune the backbone?


## Academic integrity and AI use

You may use AI assistants for debugging, explanation, and brainstorming.

You remain responsible for understanding the notebook you submit. Be prepared to explain:

- what the frozen CNN representation is
- what the head predicts
- why hue/saturation are represented as disk coordinates
- how you compared experiments
- what your plots show
